# Full Phase-0: CIFAR-10 / ResNet-18 trajectory signatures
This notebook runs the 30 training jobs on a T4 and stores every completed run plus epoch checkpoints in Google Drive. Re-running all cells resumes safely. It does **not** issue a scientific verdict.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
source_zip = Path('/content/phase0_source.zip')
if not source_zip.exists():
    print('Upload phase0_source.zip generated with this repository.')
    uploaded = files.upload()
    if 'phase0_source.zip' not in uploaded:
        raise RuntimeError('Expected phase0_source.zip')
project = Path('/content/trajectory-signature-phase0')
if project.exists():
    shutil.rmtree(project)
project.mkdir()
with zipfile.ZipFile(source_zip) as archive:
    archive.extractall(project)
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]'], check=True)
print('Installed project from', project)

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable. Select Runtime > Change runtime type > T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

In [ ]:
output = Path('/content/drive/MyDrive/trajsig_phase0_results')
output.mkdir(parents=True, exist_ok=True)
print('Persistent output:', output)
subprocess.run([sys.executable, '-m', 'trajsig', 'train', '--config', 'configs/full.yaml', '--output', str(output)], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'trajsig', 'audit', '--config', 'configs/full.yaml', '--output', str(output)], check=True)
archive = output / 'phase0_training_artifacts.zip'
subprocess.run([sys.executable, '-m', 'trajsig', 'package', '--config', 'configs/full.yaml', '--output', str(output), '--destination', str(archive)], check=True)
print('FULL TRAINING AND AUDIT COMPLETE')
print('Return this file for Stage 5 analysis:', archive)
print('Size (GiB):', archive.stat().st_size / 1024**3)